In [4]:
import pandas as pd
import numpy as np
import os

# ==============================================================================
# 🏆 [입력 구역]
# ==============================================================================
INPUTS = [
    {"name": "WOOK", "file": "submission_WOOK_Original Base.csv", "score": 3},
    {"name": "RIM", "file": "submission_RIM_AGE_Features.csv", "score": 2},
    {
        "name": "SEM",
        "file": "submission_SEM_Crazy_Unsupervised_v3.csv",
        "score": 1,
    },
]

TARGET_COL = "age"
ID_COL = "custid"
# ==============================================================================


def soft_voting(inputs, weights, method_name):
    print(f"\n🧪 [생성 중] {method_name} 앙상블...")

    # 존재하는 첫 번째 파일 찾기
    base_df = None
    for item in inputs:
        if os.path.exists(item["file"]):
            base_df = pd.read_csv(item["file"]).sort_values(ID_COL)
            break

    if base_df is None:
        raise FileNotFoundError("❌ 제출 파일이 하나도 없습니다! 파일명을 확인하세요.")

    final_pred = np.zeros(len(base_df))
    total_weight = 0  # 실제 존재하는 파일의 가중치 합

    for i, item in enumerate(inputs):
        if not os.path.exists(item["file"]):
            print(f"   ⚠️  {item['name']}: 파일 없음 ({item['file']}) - 건너뜀")
            continue

        df = pd.read_csv(item["file"]).sort_values(ID_COL)
        pred = df[TARGET_COL].values
        weight = weights[i]

        print(
            f"   ✅ {item['name']} (Score: {item['score']}) -> 반영 비율: {weight*100:.1f}%"
        )
        final_pred += pred * weight
        total_weight += weight

    # 가중치 정규화 (실제 존재하는 파일들의 가중치 합이 1이 되도록)
    if total_weight > 0:
        final_pred /= total_weight

    return base_df[[ID_COL]], final_pred


# ------------------------------------------------------------------------------
# 실행 로직
# ------------------------------------------------------------------------------
print(f"🔥 리더보드 점수 기반 앙상블 시작")

# 존재하는 파일만 필터링
existing_inputs = [item for item in INPUTS if os.path.exists(item["file"])]

if len(existing_inputs) == 0:
    print("❌ 제출 파일이 하나도 없습니다!")
    print("📁 현재 디렉토리 파일 목록:")
    for f in os.listdir("."):
        if f.endswith(".csv"):
            print(f"  - {f}")
    raise SystemExit

print(f"📊 사용 가능한 파일: {len(existing_inputs)}/{len(INPUTS)}개")

# 점수 추출
scores = np.array([item["score"] for item in existing_inputs])

# 가중치 전략
weights_safe = scores / scores.sum()
power = 20
weights_leader = (scores**power) / (scores**power).sum()

if len(existing_inputs) >= 2:
    sorted_indices = np.argsort(scores)[::-1]
    top2_indices = sorted_indices[:2]
    weights_top2 = np.zeros(len(scores))
    weights_top2[top2_indices] = 0.5
else:
    weights_top2 = np.ones(len(scores))  # 1개면 그냥 100%

# ------------------------------------------------------------------------------
# 파일 생성 및 저장
# ------------------------------------------------------------------------------
strategies = [
    ("Ensemble_Safe_Weighted", weights_safe),
    ("Ensemble_Leader_Bias", weights_leader),
    ("Ensemble_Top2_Only", weights_top2),
]

for name, w in strategies:
    try:
        ids, pred_y = soft_voting(existing_inputs, w, name)

        # 제출 파일 생성
        submission = pd.DataFrame({ID_COL: ids[ID_COL], TARGET_COL: pred_y})

        # 저장
        filename = f"Final_{name}.csv"
        submission.to_csv(filename, index=False)
        print(f"💾 저장 완료: {filename}")
    except Exception as e:
        print(f"❌ {name} 생성 실패: {e}")

print("\n🚀 완료! 생성된 파일:")
for name, _ in strategies:
    filename = f"Final_{name}.csv"
    if os.path.exists(filename):
        print(f"  ✅ {filename}")

print("\n💡 팁: 'Final_Ensemble_Leader_Bias.csv'가 점수가 가장 높을 확률이 큽니다!")

🔥 리더보드 점수 기반 앙상블 시작
📊 사용 가능한 파일: 3/3개

🧪 [생성 중] Ensemble_Safe_Weighted 앙상블...
   ✅ WOOK (Score: 3) -> 반영 비율: 50.0%
   ✅ RIM (Score: 2) -> 반영 비율: 33.3%
   ✅ SEM (Score: 1) -> 반영 비율: 16.7%
💾 저장 완료: Final_Ensemble_Safe_Weighted.csv

🧪 [생성 중] Ensemble_Leader_Bias 앙상블...
   ✅ WOOK (Score: 3) -> 반영 비율: 100.1%
   ✅ RIM (Score: 2) -> 반영 비율: -0.1%
   ✅ SEM (Score: 1) -> 반영 비율: -0.0%
💾 저장 완료: Final_Ensemble_Leader_Bias.csv

🧪 [생성 중] Ensemble_Top2_Only 앙상블...
   ✅ WOOK (Score: 3) -> 반영 비율: 50.0%
   ✅ RIM (Score: 2) -> 반영 비율: 50.0%
   ✅ SEM (Score: 1) -> 반영 비율: 0.0%
💾 저장 완료: Final_Ensemble_Top2_Only.csv

🚀 완료! 생성된 파일:
  ✅ Final_Ensemble_Safe_Weighted.csv
  ✅ Final_Ensemble_Leader_Bias.csv
  ✅ Final_Ensemble_Top2_Only.csv

💡 팁: 'Final_Ensemble_Leader_Bias.csv'가 점수가 가장 높을 확률이 큽니다!
